In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import joblib
from tqdm import tqdm

DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
NOISE_AMP = 0.0005
NOISE_OFFSET = 1.0

WEIGHTS = torch.tensor([10.0, 1.0, 1.0]).to(DEVICE)

class TinyBlobNet(nn.Module):
    def __init__(self, input_size=400, output_size=3, intermediate_size=300):
        super().__init__()

        # self.conv1 = nn.Conv2d(1, 1, 3, 1, 1)
        # self.conv2 = nn.Conv2d(8, 1, 3, 1, 1)

        self.full_conn = nn.Sequential(
            nn.Linear(input_size, intermediate_size),
            nn.GELU(),
            nn.Linear(intermediate_size, intermediate_size * 2),
            # nn.Linear(intermediate_size, intermediate_size),
            nn.GELU(),
            nn.Linear(intermediate_size * 2, intermediate_size),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(intermediate_size, output_size)
            # nn.BatchNorm2d(400),
            # nn.Linear(input_size, intermediate_size),
            # nn.ReLU(),
            # # nn.Dropout(0.3),
            # nn.Linear(intermediate_size, intermediate_size),
            # nn.ReLU(),
            # # # nn.Linear(intermediate_size, 150),
            # # nn.ReLU(),
            # nn.Linear(intermediate_size, output_size),

            # nn.ReLU(),
            # nn.Linear(intermediate_size, output_size),
        )

    def forward(self, x):
        # x = self.conv1(x)
        # x = self.conv2(x)
        x = x.view(x.size(0), -1)
        return self.full_conn(x)


class BlobDataset(Dataset):
    def __init__(self, noisy_file, clean_file):
        clean_dat = joblib.load(clean_file)
        noisy_dat = joblib.load(noisy_file)
        
        self.noisy = torch.from_numpy(noisy_dat).float().reshape(-1, 1, 20, 20)
        self.clean = torch.from_numpy(clean_dat["blobs"]).float().reshape(-1, 1, 20, 20)
        
        self.targets = torch.cat([
            torch.from_numpy(clean_dat["integrals"]).float().unsqueeze(1),
            torch.from_numpy(clean_dat["cents"]).float()
        ], dim=1)

    def __len__(self): 
        return len(self.clean)
        
    def __getitem__(self, idx): 
        return self.noisy[idx], self.clean[idx], self.targets[idx]

def weighted_mse_loss(input, target, weights):
    return ((input - target) ** 2 * weights).sum(dim=1).mean()


if __name__ == "__main__":
    train_ds = BlobDataset("TRA_CLEAN_DAT.joblib", "TRA_CLEAN_DAT.joblib")
    val_ds = BlobDataset("VAL_NOISY_DAT.joblib", "VAL_CLEAN_DAT.joblib")

    train_loader = DataLoader(train_ds, 
                            batch_size=1024, 
                            shuffle=True,
                            num_workers=0,             
                            )


    val_loader = DataLoader(val_ds, batch_size=1024, 
                            shuffle=False, 
                            num_workers=0,             
                            )

    model = TinyBlobNet().to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5)

    best_val = float('inf')


    for epoch in range(80):
        model.train()
        t_loss = 0.0
        
        for _, clean_orig, targets_orig in tqdm(train_loader, desc=f"Epoch {epoch+1}", leave=False):

            optimizer.zero_grad(set_to_none=True)
            
            clean_orig = clean_orig.to(DEVICE)
            targets_orig = targets_orig.to(DEVICE)

            clean = clean_orig.clone()
            targets = targets_orig.clone()

            with torch.no_grad():
                if torch.rand(1).item() > 0.5:
                    clean = torch.flip(clean, dims=[3]) 
                    targets[:, 1] = 19.0 - targets[:, 1]
                if torch.rand(1).item() > 0.5:
                    clean = torch.flip(clean, dims=[2]) 
                    targets[:, 2] = 19.0 - targets[:, 2]
                
            
                sx, sy = torch.randint(-2, 3, (1,)).item(), torch.randint(-2, 3, (1,)).item()
                clean = torch.roll(clean, shifts=(sy, sx), dims=(2, 3))
                targets[:, 1] += sx
                targets[:, 2] += sy
                targets[:, 1:] = torch.clamp(targets[:, 1:], 0.0, 19.0)

        
            noise = (torch.rand(clean.shape, device=DEVICE) * 2 - 1) * NOISE_AMP
        
            off_val = torch.rand((clean.shape[0], 1, 1, 1), device=DEVICE) * NOISE_OFFSET
            noisy_inputs = clean + noise + off_val
            
    
            preds = model(noisy_inputs)
            loss = weighted_mse_loss(preds, targets, WEIGHTS)
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            t_loss += loss.item()

      
        model.eval()
        v_loss = 0.0
        with torch.no_grad():
            for noisy, _, v_targets in val_loader:
                noisy = noisy.to(DEVICE)
                v_targets = v_targets.to(DEVICE)

                v_preds = model(noisy)
                v_loss += weighted_mse_loss(v_preds, v_targets, WEIGHTS).item()

        avg_t = t_loss / len(train_loader)
        avg_v = v_loss / len(val_loader)
        scheduler.step(avg_v)
        
        current_lr = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch+1:02d} | Train Loss: {avg_t:.6f} | Val Loss: {avg_v:.6f} | LR: {current_lr:.2e}")

   
        if avg_v < best_val:
            best_val = avg_v
            torch.save(model.state_dict(), "best_tiny_model.pt")
            print(f"  --> Saved new best model!")

    print("\nTraining Complete. Best Validation Loss:", best_val)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Total parameters: {total_params:,}")